# 04. Q&A 파이프라인 데모

7-의도 라우터를 통해 다양한 질문 유형에 답변하는 데모입니다.

**소요 시간**: 약 5분 (예시 질문 7개)

## 4.1 환경 설정

In [ ]:
import os, sys
from google.colab import userdata
sys.path.insert(0, '/content/repo')

# 환경변수
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    os.environ['CLOVA_API_KEY'] = userdata.get('CLOVA_API_KEY')
    os.environ['NAVER_CLIENT_ID'] = userdata.get('NAVER_CLIENT_ID')
    os.environ['NAVER_CLIENT_SECRET'] = userdata.get('NAVER_CLIENT_SECRET')
    print("모든 API 키 로드")
except Exception as e:
    print(f"Colab Secrets 누락: {e}")
    raise

## 4.2 파이프라인 초기화

In [ ]:
from src.pipeline import QAPipeline

# 파이프라인 로드 (첫 호출 시 reranker GPU 로딩 ~70초)
pipeline = QAPipeline(
    facts_db='/content/db/facts.db',
    chroma_path='/content/db/chroma_rag',
    use_gpu=True,  # GPU 가능하면 True
)
print("파이프라인 준비 완료")

## 4.3 7-의도 데모 (각 의도별 1개씩)

In [ ]:
# 1. fact_lookup — 정형 수치 조회
result = pipeline.ask("삼성전자 2025년 매출은?")
print(f"[fact_lookup] {result['intent']}")
print(f"  {result['elapsed']:.1f}초")
print(f" {result['answer']}")
print()

In [ ]:
# 2. narrative — 비정형 서술 검색
result = pipeline.ask("삼성전자의 사업 부문은 어떻게 구성되어 있나요?")
print(f" [narrative] {result['intent']}")
print(f"  {result['elapsed']:.1f}초")
print(f" {result['answer'][:300]}...")
print()

In [ ]:
# 3. definition — 용어 사전
result = pipeline.ask("PER이 뭐야?")
print(f" [definition] {result['intent']}")
print(f"  {result['elapsed']:.1f}초")
print(f" {result['answer']}")
print()

In [ ]:
# 4. sector_compare — 업종 비교
result = pipeline.ask("반도체 업종 비교해줘")
print(f" [sector_compare] {result['intent']}")
print(f"  {result['elapsed']:.1f}초")
print(f" {result['answer'][:400]}...")
print()

In [ ]:
# 5. news_context — 실시간 뉴스 + 공시 통합
result = pipeline.ask("삼성전자 최근 동향이 어떤가요?")
print(f" [news_context] {result['intent']}")
print(f"  {result['elapsed']:.1f}초")
print(f" {result['answer'][:400]}...")
print()

## 4.4 챗봇 메모리 데모 (후속 질문)

In [ ]:
# 같은 세션 ID로 연속 질문 → 메모리 자동 보충
SESSION = 'demo-session-001'

# Q1
r1 = pipeline.ask("삼성전자 2025년 매출은?", session_id=SESSION)
print(f"Q1: 삼성전자 2025년 매출은?")
print(f"A1: {r1['answer']}\n")

# Q2 — 회사명 생략 + "전년 대비"
r2 = pipeline.ask("전년 대비는?", session_id=SESSION)
print(f"Q2: 전년 대비는?")
print(f"A2: {r2['answer']}")
print(f"   (메모리: 회사='삼성전자', 계정='매출액' 자동 보충)\n")

# Q3 — 계정만 변경
r3 = pipeline.ask("그럼 영업이익은?", session_id=SESSION)
print(f"Q3: 그럼 영업이익은?")
print(f"A3: {r3['answer']}")
print(f"   (메모리: 회사='삼성전자', 연도='2024' 유지, 계정만 변경)")

## 4.5 다음 단계

45개 질의로 정량 평가를 실행합니다.

→ **05_evaluation.ipynb** 로 진행